## Deliverables

In [88]:
import torch
import torch.nn as nn
import tiktoken
import re
import torch.nn.functional as F

In [89]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # embedding dimension
    "n_heads": 12,           # number of attention heads
    "n_layers": 12,          # number of transformer blocks
    "drop_rate": 0.1,        # dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

In [90]:
# Layer normalization

def layer_norm(x, scale, shift, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + eps)
    return scale * norm_x + shift


# Compare with PyTorch's built-in version

# define parameters
torch.manual_seed(42)
batch = torch.randn(2, 5, GPT_CONFIG_124M["emb_dim"])
scale = nn.Parameter(torch.ones(GPT_CONFIG_124M["emb_dim"]))
shift = nn.Parameter(torch.zeros(GPT_CONFIG_124M["emb_dim"]))

# pytorch
pytorch_ln = nn.LayerNorm(GPT_CONFIG_124M["emb_dim"])
pytorch_ln.weight = nn.Parameter(scale)
pytorch_ln.bias = nn.Parameter(shift)

# custom layer norm
out_custom = layer_norm(batch, scale, shift)
out_pytorch = pytorch_ln(batch)

print("Custom Layer Normalization")
print(out_custom)

print("Pytorch Normalization")
print(out_pytorch)

Custom Layer Normalization
tensor([[[ 1.8952,  1.4546,  0.8668,  ..., -1.6426, -0.4665,  0.5416],
         [ 0.3753, -3.1052, -1.4458,  ...,  0.2497,  0.3557,  1.3597],
         [-0.8698, -0.7143, -0.2062,  ..., -0.0257,  1.3141,  1.1219],
         [-1.0159, -0.1616, -0.5069,  ..., -2.3742,  0.6913, -1.5789],
         [-0.9686, -1.0593, -0.6165,  ..., -1.5553, -1.4347,  0.1986]],

        [[-0.1416,  0.0713,  0.6640,  ..., -1.1238,  1.2655,  0.9075],
         [ 0.6808,  0.3704, -0.9783,  ..., -1.2212, -0.2140,  0.1534],
         [-0.6430, -1.0830,  0.3806,  ...,  0.1340,  0.2137,  0.7687],
         [-0.2970,  1.3508,  0.7984,  ..., -0.6203, -1.3819, -1.6725],
         [-0.4373,  2.2695,  1.9999,  ..., -0.8034,  0.6053, -0.8071]]],
       grad_fn=<AddBackward0>)
Pytorch Normalization
tensor([[[ 1.8952,  1.4546,  0.8668,  ..., -1.6426, -0.4665,  0.5416],
         [ 0.3753, -3.1052, -1.4458,  ...,  0.2497,  0.3557,  1.3597],
         [-0.8698, -0.7143, -0.2062,  ..., -0.0257,  1.3141,  1.

In [91]:
# GELU Implementation

def gelu(x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
    ))

# Compare with ReLU behavior

# RELU from PyTorch
x = torch.linspace(-3, 3, 200)
relu = nn.ReLU()

custom_gelu = gelu(x)
pytorch_relu = relu(x)

print("Custom GELU")
print(custom_gelu)

print("Pytorch RELU")
print(pytorch_relu)

Custom GELU
tensor([-0.0036, -0.0040, -0.0044, -0.0048, -0.0053, -0.0058, -0.0063, -0.0069,
        -0.0075, -0.0082, -0.0089, -0.0097, -0.0105, -0.0114, -0.0124, -0.0134,
        -0.0144, -0.0156, -0.0168, -0.0181, -0.0194, -0.0209, -0.0224, -0.0240,
        -0.0257, -0.0274, -0.0293, -0.0312, -0.0333, -0.0354, -0.0377, -0.0400,
        -0.0424, -0.0450, -0.0476, -0.0503, -0.0531, -0.0561, -0.0591, -0.0622,
        -0.0654, -0.0687, -0.0720, -0.0755, -0.0790, -0.0826, -0.0863, -0.0900,
        -0.0937, -0.0975, -0.1014, -0.1053, -0.1091, -0.1130, -0.1169, -0.1208,
        -0.1246, -0.1284, -0.1321, -0.1358, -0.1394, -0.1428, -0.1462, -0.1494,
        -0.1524, -0.1553, -0.1580, -0.1604, -0.1627, -0.1646, -0.1663, -0.1677,
        -0.1688, -0.1696, -0.1700, -0.1700, -0.1696, -0.1688, -0.1676, -0.1658,
        -0.1636, -0.1609, -0.1577, -0.1540, -0.1496, -0.1447, -0.1392, -0.1331,
        -0.1264, -0.1190, -0.1109, -0.1022, -0.0928, -0.0828, -0.0720, -0.0605,
        -0.0483, -0.0354, -0

In [92]:
# Feed-Forward Network with gelu

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(GPT_CONFIG_124M["emb_dim"], 4 * GPT_CONFIG_124M["emb_dim"])
        self.linear2 = nn.Linear(4 * GPT_CONFIG_124M["emb_dim"], GPT_CONFIG_124M["emb_dim"])

    def forward(self, x):
        x = self.linear1(x)
        x = gelu(x)
        x = self.linear2(x)
        return x


# Understand expansion ratio (4x hidden size)
num_dims = GPT_CONFIG_124M["emb_dim"]
ffn = FeedForward()
sample = torch.randn(1, 5, num_dims)

# Step thorugh
step1 = ffn.linear1(sample)
print("Step 1", step1.size(), "\n")
step2 = gelu(step1)
print("Step 2", step2.size(), "\n")
step3 = ffn.linear2(step2)
print("Step 3", step3.size(), "\n")

print(f"\nExpansion ratio: {4 * num_dims} / {num_dims} = {4 * num_dims // num_dims}x")

Step 1 torch.Size([1, 5, 3072]) 

Step 2 torch.Size([1, 5, 3072]) 

Step 3 torch.Size([1, 5, 768]) 


Expansion ratio: 3072 / 768 = 4x


In [93]:
# TransformerBlock with FFN

# def transformer_block(x, att, norm1_scale, norm1_shift, norm2_scale, norm2_shift, drop_rate, training):

#   ff = FeedForward()

#   # Attention block
#   shortcut = x
#   x = layer_norm(x, norm1_scale, norm1_shift)
#   x = att(x)
#   x = torch.nn.functional.dropout(x, p=drop_rate, training=training)
#   # residual
#   x = x + shortcut

#   # FFN block
#   shortcut = x
#   # Layer norm
#   x = layer_norm(x, norm2_scale, norm2_shift)
#   x = ff(x)
#   # dropout
#   x = torch.nn.functional.dropout(x, p=drop_rate, training=training)
#   # residual
#   x = x + shortcut

#   return x

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward()
        self.norm1_scale = nn.Parameter(torch.ones(cfg["emb_dim"]))
        self.norm1_shift = nn.Parameter(torch.zeros(cfg["emb_dim"]))
        self.norm2_scale = nn.Parameter(torch.ones(cfg["emb_dim"]))
        self.norm2_shift = nn.Parameter(torch.zeros(cfg["emb_dim"]))
        self.drop_rate = cfg["drop_rate"]

    def forward(self, x):
        # Attention block
        shortcut = x
        x = layer_norm(x, self.norm1_scale, self.norm1_shift)
        x = self.att(x)
        x = torch.nn.functional.dropout(x, p=self.drop_rate, training=self.training)
        x = x + shortcut

        # FFN block
        shortcut = x
        x = layer_norm(x, self.norm2_scale, self.norm2_shift)
        x = self.ff(x)
        x = torch.nn.functional.dropout(x, p=self.drop_rate, training=self.training)
        x = x + shortcut

        return x


In [94]:
# Test transformer block with MHA from chapter 3

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


# --------------------------------------------------------------------------------

torch.manual_seed(42)
cfg = GPT_CONFIG_124M

sample_input = torch.randn(2, 5, cfg["emb_dim"])

# TransformerBlock
block = TransformerBlock(cfg)
block_out = block(sample_input)

print("Input shape: ", sample_input.shape)
print("Output shape: ", block_out.shape)

Input shape:  torch.Size([2, 5, 768])
Output shape:  torch.Size([2, 5, 768])


In [95]:
# GPT Implementation - copied from book and edited

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])


    self.final_norm_scale = nn.Parameter(torch.ones(cfg["emb_dim"]))
    self.final_norm_shift = nn.Parameter(torch.zeros(cfg["emb_dim"]))

    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = layer_norm(x, self.final_norm_scale, self.final_norm_shift)
    logits = self.out_head(x)
    return logits



In [96]:
# Test Untrained GPT Model

# Referenced from book
def generate_text(model, idx, num_tokens):
  for _ in range(num_tokens):
    idx_cond = idx[:, -GPT_CONFIG_124M["context_length"]:]

    with torch.no_grad():
        logits = model(idx_cond)

    logits = logits[:, -1, :]
    idx_next = torch.argmax(logits, dim=-1, keepdim=True)
    idx = torch.cat((idx, idx_next), dim=1)

  return idx


# Using tiktoken

torch.manual_seed(12)
model = GPTModel(GPT_CONFIG_124M)

tokenizer_tiktoken = tiktoken.get_encoding("gpt2")

start_text = "Hello! How are you?"
encoded = tokenizer_tiktoken.encode(start_text)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

print("Prompt: ", start_text)
print("Encoded: ", encoded)

out = generate_text(model, encoded_tensor, num_tokens=15)

# Decode text
decoded = tokenizer_tiktoken.decode(out.squeeze(0).tolist())


print("Output: ", decoded)

Prompt:  Hello! How are you?
Encoded:  [15496, 0, 1374, 389, 345, 30]
Output:  Hello! How are you?The...) Guys Roche've chase Sag represented 175 veterinarySR Academy MIL Duo Symptoms


In [97]:
# Model size calculation and parameter count breakdown.
# Code referenced from book

model = GPTModel(GPT_CONFIG_124M)
total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")


# Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_bytes = total_params_gpt2 * 4

# Convert to megabytes
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")



Number of trainable parameters considering weight tying: 842,447,360
Total size of the model: 3213.68 MB


## Exercies

In [98]:
# Used config sizes from book

# GPT2_SMALL
GPT2_SMALL = {
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.1,
    "qkv_bias": False
}

# GPT2_MEDIUM
GPT2_MEDIUM = {
    "emb_dim": 1024,
    "n_layers": 24,
    "n_heads": 16,
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.1,
    "qkv_bias": False
}

# GPT2_LARGE
GPT2_LARGE = {
    "emb_dim": 1280,
    "n_layers": 36,
    "n_heads": 20,
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.1,
    "qkv_bias": False
}

# GPT2_XL
GPT2_XL = {
    "emb_dim": 1600,
    "n_layers": 48,
    "n_heads": 25,
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.1,
    "qkv_bias": False
}

sizes = {
    "GPT2-small": GPT2_SMALL,
    "GPT2-medium": GPT2_MEDIUM,
    "GPT2-large": GPT2_LARGE,
    "GPT2-XL": GPT2_XL,
}

In [99]:
# Exercise 4.1: Implement different GPT sizes (small: 6 layers, medium: 12 layers) and compare

configs = {
    "Small (6 layers)": {**GPT_CONFIG_124M, "n_layers": 6},
    "Medium (12 layers)": {**GPT_CONFIG_124M, "n_layers": 12},
}

tokenizer_tiktoken = tiktoken.get_encoding("gpt2")

start_text = "Hello! How are you?"
encoded = tokenizer_tiktoken.encode(start_text)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

for name, cfg in configs.items():
    torch.manual_seed(42)
    model = GPTModel(cfg)

    total = sum(p.numel() for p in model.parameters())
    tied = total - sum(p.numel() for p in model.out_head.parameters())
    size_mb = tied * 4 / (1024 ** 2)

    print(name)
    print(f"Model Size: {size_mb:.1f} MB")
    print(f"Output shape: {logits.shape}")
    print("\n")

Small (6 layers)
Model Size: 312.4 MB
Output shape: torch.Size([2, 4, 50257])


Medium (12 layers)
Model Size: 474.6 MB
Output shape: torch.Size([2, 4, 50257])




In [100]:
# Exercise 4.4: Analyze memory usage for different model sizes

for name, cfg in sizes.items():
    model = GPTModel(cfg)
    total_params = sum(p.numel() for p in model.parameters())
    tied_params = total_params - sum(p.numel() for p in model.out_head.parameters())
    size_mb = tied_params * 4 / (1024 * 1024)

    print(f"Model: {name}")
    print(f"Total Params: {tied_params:,}")
    print(f"Model SIze: {size_mb:.2f} MB")
    print()
    del model

Model: GPT2-small
Total Params: 124,412,160
Model SIze: 474.59 MB

Model: GPT2-medium
Total Params: 266,638,336
Model SIze: 1017.14 MB

Model: GPT2-large
Total Params: 471,809,792
Model SIze: 1799.81 MB

Model: GPT2-XL
Total Params: 800,633,536
Model SIze: 3054.17 MB

